# 04 — Baseline Models
## Airline Flight Delay Prediction Project

---

### Purpose of this notebook

Before building a deep neural network, we establish **strong baselines**.
This is one of the most important habits in professional machine learning:

> *"A complex model that barely beats a simple one is not worth deploying."*

Baselines serve three purposes:

1. **Sanity check** — if Linear Regression already achieves R²=0.90, a DNN adds little value
2. **Performance floor** — every model we build must beat the best baseline to justify its complexity
3. **Feature validation** — if ALL models perform poorly, the problem is the features, not the architecture

---

### Models in this notebook

| Model | Why we include it |
|---|---|
| **Dummy (mean predictor)** | Absolute floor — predicts the training mean every time |
| **Linear Regression** | Tests if a linear relationship exists between features and target |
| **Ridge Regression** | Linear Regression + L2 regularisation — more robust, less overfitting |
| **Random Forest** | Ensemble of decision trees — captures non-linear interactions |
| **Gradient Boosting** | Sequential ensemble — often the best tree-based model |

---

> **Input:** `data/processed/` — numpy arrays from notebook 03  
> **Output:** `reports/figures/` — all comparison plots  
> **Key output:** A results table we will update in notebook 05 with the DNN score

---
*Notebook: `04_baseline_models.ipynb` | Project: Airline Delay DNN*

---
## 1. Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────
import warnings, json, pickle
warnings.filterwarnings('ignore')

# ── Numerics ──────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Scikit-learn models ───────────────────────────────────────
from sklearn.linear_model   import LinearRegression, Ridge
from sklearn.ensemble       import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics        import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection     import permutation_importance

# ── Optional: XGBoost (install with: pip install xgboost) ─────
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
    print("✅ XGBoost available")
except ImportError:
    HAS_XGBOOST = False
    print("⚠️  XGBoost not installed — using GradientBoostingRegressor instead")
    print("   Install with: pip install xgboost")

# ── Plot settings ─────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120, 'figure.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
})

OUT_PATH = '../data/processed/'
FIG_PATH = '../reports/figures/'

FEATURE_COLS = [
    'month_sin', 'month_cos', 'year_norm', 'log_arr_flights',
    'cancel_rate', 'divert_rate', 'carrier_hist_delay', 'airport_hist_delay',
    'is_summer', 'is_holiday_month', 'carrier_enc', 'airport_enc'
]
TARGET_COL = 'delay_rate'

print(f"\n✅ Configuration set — {len(FEATURE_COLS)} features")

---
## 2. Load Preprocessed Arrays

We load the six numpy arrays saved by notebook 03.
No feature engineering or scaling needed — it's already done.

In [ ]:
# ── Load preprocessed arrays ──────────────────────────────────
X_train = np.load(OUT_PATH + 'X_train.npy')
X_val   = np.load(OUT_PATH + 'X_val.npy')
X_test  = np.load(OUT_PATH + 'X_test.npy')
y_train = np.load(OUT_PATH + 'y_train.npy')
y_val   = np.load(OUT_PATH + 'y_val.npy')
y_test  = np.load(OUT_PATH + 'y_test.npy')

print("Loaded arrays:")
print(f"  X_train : {X_train.shape}    y_train : {y_train.shape}")
print(f"  X_val   : {X_val.shape}     y_val   : {y_val.shape}")
print(f"  X_test  : {X_test.shape}    y_test  : {y_test.shape}")
print(f"\n  Null check — X_train: {np.isnan(X_train).sum()} | "
      f"X_val: {np.isnan(X_val).sum()} | X_test: {np.isnan(X_test).sum()}")

---
## 3. Evaluation Framework

### Metrics we use

| Metric | Formula | What it measures |
|---|---|---|
| **MAE** | `mean(|y - ŷ|)` | Average absolute error — same units as target (delay rate) |
| **RMSE** | `sqrt(mean((y-ŷ)²))` | Penalises large errors more — sensitive to outliers |
| **R²** | `1 - SS_res/SS_tot` | Proportion of variance explained — 1.0 = perfect, 0.0 = mean predictor |

### Why MAE is our primary metric

Our target `delay_rate` has some high-value outliers (airports with 100% delay rate).
- **RMSE** penalises these outliers heavily, which can mislead comparisons
- **MAE** treats all errors equally — more representative of typical prediction quality
- **R²** gives intuition but is sensitive to the split's variance

We report all three but rank models primarily by **validation MAE**.

In [ ]:
# ── Evaluation helper function ────────────────────────────────
def evaluate_model(name, model, X_tr, y_tr, X_vl, y_vl, X_te, y_te):
    """
    Train a model, evaluate on val and test, return a results dict.

    Parameters
    ----------
    name   : str — display name for the model
    model  : sklearn estimator with fit() and predict()
    X_tr, y_tr : training arrays
    X_vl, y_vl : validation arrays
    X_te, y_te : test arrays

    Returns
    -------
    dict with keys: name, val_mae, val_rmse, val_r2, test_mae, test_rmse, test_r2
    """
    # ── Train ──────────────────────────────────────────────────
    model.fit(X_tr, y_tr)

    # ── Predict on val and test ────────────────────────────────
    pred_val  = model.predict(X_vl)
    pred_test = model.predict(X_te)

    # ── Clip predictions to valid range [0, 1] ─────────────────
    # Some linear models may predict negative values or values > 1
    # Physical constraint: delay_rate must be in [0, 1]
    pred_val  = np.clip(pred_val,  0.0, 1.0)
    pred_test = np.clip(pred_test, 0.0, 1.0)

    # ── Compute metrics ────────────────────────────────────────
    results = {
        'name'      : name,
        'val_mae'   : mean_absolute_error(y_vl, pred_val),
        'val_rmse'  : np.sqrt(mean_squared_error(y_vl, pred_val)),
        'val_r2'    : r2_score(y_vl, pred_val),
        'test_mae'  : mean_absolute_error(y_te, pred_test),
        'test_rmse' : np.sqrt(mean_squared_error(y_te, pred_test)),
        'test_r2'   : r2_score(y_te, pred_test),
        'pred_val'  : pred_val,
        'pred_test' : pred_test,
        'model'     : model,
    }

    print(f"  {name:<30} "
          f"Val  MAE={results['val_mae']:.4f}  RMSE={results['val_rmse']:.4f}  R²={results['val_r2']:.4f}  |  "
          f"Test MAE={results['test_mae']:.4f}  RMSE={results['test_rmse']:.4f}  R²={results['test_r2']:.4f}")
    return results


# ── Store all results ─────────────────────────────────────────
all_results = []   # list of result dicts — filled by each model cell
print("✅ evaluate_model() ready")

---
## 4. Model 1 — Dummy Baseline (Mean Predictor)

### What it does

This model ignores all features and **always predicts the training mean**:

```
ŷ = mean(y_train) = 0.2074
```

### Why it matters

This is the **absolute floor**. If your model can't beat this, you have:
- A feature engineering problem
- A data quality problem
- Or the task is genuinely unpredictable

Any model with Val MAE > 0.0989 is performing *worse than doing nothing*.

### Expected R²

A mean predictor has R² = 0 on the training set by definition.
On val/test it will be negative if the val/test variance differs from train
(which it does — remember the COVID distribution shift in 2020).

In [ ]:
# ── Dummy baseline: predict training mean for every sample ────
train_mean = y_train.mean()
print(f"Training mean (y_train): {train_mean:.6f}")

# Evaluate manually (sklearn's DummyRegressor does the same thing)
dummy_pred_val  = np.full(len(y_val),  train_mean)
dummy_pred_test = np.full(len(y_test), train_mean)

dummy_results = {
    'name'      : 'Dummy (mean)',
    'val_mae'   : mean_absolute_error(y_val,  dummy_pred_val),
    'val_rmse'  : np.sqrt(mean_squared_error(y_val, dummy_pred_val)),
    'val_r2'    : r2_score(y_val, dummy_pred_val),
    'test_mae'  : mean_absolute_error(y_test, dummy_pred_test),
    'test_rmse' : np.sqrt(mean_squared_error(y_test, dummy_pred_test)),
    'test_r2'   : r2_score(y_test, dummy_pred_test),
    'pred_val'  : dummy_pred_val,
    'pred_test' : dummy_pred_test,
    'model'     : None,
}
all_results.append(dummy_results)

print(f"\nDummy Baseline Results:")
print(f"  Val  — MAE: {dummy_results['val_mae']:.4f}  "
      f"RMSE: {dummy_results['val_rmse']:.4f}  R²: {dummy_results['val_r2']:.4f}")
print(f"  Test — MAE: {dummy_results['test_mae']:.4f}  "
      f"RMSE: {dummy_results['test_rmse']:.4f}  R²: {dummy_results['test_r2']:.4f}")
print(f"\n⚠️  Negative R² on val is expected (COVID-2020 distribution shift)")
print(f"   Val mean={y_val.mean():.4f} vs Train mean={y_train.mean():.4f}")

---
## 5. Model 2 — Linear Regression

### What it learns

Linear Regression fits a hyperplane:

```
ŷ = β₀ + β₁·month_sin + β₂·month_cos + β₃·year_norm + ... + β₁₂·airport_enc
```

It finds the weights `β` that minimise the sum of squared residuals (OLS).

### What this test tells us

- If Linear Regression performs well → the relationship between features and target is approximately **linear**
- If it performs poorly → there are **non-linear interactions** that require a more complex model
- The gap between Linear Regression and Random Forest reveals how much non-linearity matters

### Interpretation of coefficients

With `StandardScaler` applied, the coefficients are directly comparable:
a larger absolute value means the feature has a stronger linear relationship with delay_rate.

In [ ]:
# ── Linear Regression ─────────────────────────────────────────
print("Training Linear Regression...")
lr = LinearRegression()
lr_results = evaluate_model(
    'Linear Regression', lr,
    X_train, y_train, X_val, y_val, X_test, y_test
)
all_results.append(lr_results)

In [ ]:
# ── Inspect coefficients ─────────────────────────────────────
coef_df = pd.DataFrame({
    'feature'    : FEATURE_COLS,
    'coefficient': lr.coef_,
    'abs_coef'   : np.abs(lr.coef_)
}).sort_values('abs_coef', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['tomato' if c < 0 else 'steelblue' for c in coef_df['coefficient']]
bars = ax.barh(coef_df['feature'], coef_df['coefficient'],
               color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', lw=0.8)

# Value labels
for bar, val in zip(bars, coef_df['coefficient']):
    xpos = val + (0.001 if val >= 0 else -0.001)
    ha   = 'left' if val >= 0 else 'right'
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:+.4f}', va='center', ha=ha, fontsize=8)

ax.set_xlabel('Coefficient value (features are standardised → comparable)')
ax.set_title('Linear Regression Coefficients\n'
             'Positive = increases delay rate  |  Negative = decreases delay rate')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_PATH + '27_lr_coefficients.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 27_lr_coefficients.png")
print("\nTop coefficients:")
print(coef_df[['feature','coefficient']].head(6).to_string(index=False))

---
## 6. Model 3 — Ridge Regression (L2 Regularisation)

### What is regularisation?

Linear Regression can **overfit** — especially when features are correlated
(which ours are: `month_sin` and `month_cos` are correlated; `carrier_hist_delay`
and `airport_hist_delay` are correlated with each other).

Ridge adds an L2 penalty to the loss function:

```
Loss = Σ(y - ŷ)² + α × Σ(βᵢ²)
```

The penalty term `α × Σ(βᵢ²)` **shrinks** all coefficients toward zero,
reducing variance at the cost of a small increase in bias.

### Choosing α with cross-validation

We test multiple α values using **5-fold cross-validation on the training set**
to find the one that minimises validation MAE. This is a proper hyperparameter
search — we never use the test set to pick α.

In [ ]:
# ── Ridge: cross-validate to find best alpha ──────────────────
alphas    = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
cv_scores = []

print("5-fold CV on Ridge (MAE) per alpha:")
for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    # negative because sklearn maximises, but we want to minimise MAE
    cv = cross_val_score(ridge, X_train, y_train, cv=5,
                         scoring='neg_mean_absolute_error', n_jobs=-1)
    mean_mae  = -cv.mean()
    std_mae   = cv.std()
    cv_scores.append({'alpha': alpha, 'cv_mae': mean_mae, 'cv_std': std_mae})
    print(f"  α={alpha:<8}  CV MAE = {mean_mae:.4f} ± {std_mae:.4f}")

cv_df     = pd.DataFrame(cv_scores)
best_alpha = cv_df.loc[cv_df['cv_mae'].idxmin(), 'alpha']
print(f"\n✅ Best α = {best_alpha}")

In [ ]:
# ── Visualise CV MAE vs alpha ─────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(cv_df['alpha'], cv_df['cv_mae'],
            yerr=cv_df['cv_std'], marker='o',
            color='steelblue', lw=2, capsize=5, capthick=1.5)
ax.axvline(best_alpha, color='tomato', ls='--', lw=1.5,
           label=f'Best α={best_alpha}')
ax.set_xscale('log')
ax.set_xlabel('α (log scale)')
ax.set_ylabel('5-fold CV MAE')
ax.set_title('Ridge Regression: Cross-Validation MAE vs Regularisation Strength')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_PATH + '28_ridge_cv_alpha.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 28_ridge_cv_alpha.png")

In [ ]:
# ── Train best Ridge and evaluate ─────────────────────────────
print("Training Ridge with best alpha...")
ridge = Ridge(alpha=best_alpha)
ridge_results = evaluate_model(
    f'Ridge (α={best_alpha})', ridge,
    X_train, y_train, X_val, y_val, X_test, y_test
)
all_results.append(ridge_results)

---
## 7. Model 4 — Random Forest

### Why Random Forest?

Linear models can only learn linear relationships. But real-world delay patterns
are non-linear:
- The effect of summer is not simply additive — it **amplifies** the carrier effect
- A busy airport in summer is much worse than busy airport + summer separately

**Random Forest** handles non-linearity naturally through decision tree splits.
It also provides **feature importance** scores — a free interpretability tool.

### How it works (brief)

1. Build `n` decision trees, each on a **bootstrapped** sample of training rows
2. At each split, consider only a **random subset** of features
3. Final prediction = **average** of all tree predictions

The randomness (bagging + feature sampling) reduces variance and prevents overfitting.

### Hyperparameters we set

| Param | Value | Reasoning |
|---|---|---|
| `n_estimators` | 200 | More trees = more stable, diminishing returns after ~200 |
| `max_depth` | 12 | Deep enough to capture interactions, prevents extreme overfitting |
| `min_samples_leaf` | 20 | Each leaf must contain ≥20 samples — smooths noisy leaves |
| `max_features` | `'sqrt'` | Standard practice: sqrt(n_features) ≈ 3–4 features per split |
| `random_state` | 42 | Reproducibility |

In [ ]:
# ── Random Forest ─────────────────────────────────────────────
print("Training Random Forest (200 trees, max_depth=12)...")
print("This may take 1–2 minutes...")

rf = RandomForestRegressor(
    n_estimators   = 200,      # 200 trees — good balance of quality vs speed
    max_depth      = 12,       # cap depth to prevent overfitting
    min_samples_leaf = 20,     # smooth leaves — each needs ≥20 training samples
    max_features   = 'sqrt',   # sqrt(12) ≈ 3–4 features considered per split
    n_jobs         = -1,       # use all CPU cores
    random_state   = 42
)
rf_results = evaluate_model(
    'Random Forest', rf,
    X_train, y_train, X_val, y_val, X_test, y_test
)
all_results.append(rf_results)

In [ ]:
# ── Random Forest feature importance ──────────────────────────
# Mean decrease in impurity (MDI) — how much each feature reduces the
# weighted impurity across all splits where it is used

importance_df = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5))
palette = plt.cm.viridis(np.linspace(0.2, 0.9, len(importance_df)))
bars = ax.barh(importance_df['feature'], importance_df['importance'],
               color=palette, edgecolor='white', height=0.6)
ax.invert_yaxis()

# Add value labels
for bar, val in zip(bars, importance_df['importance']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('Mean Decrease in Impurity (normalised)')
ax.set_title('Random Forest Feature Importance\n'
             '(higher = more useful for predicting delay_rate)')
plt.tight_layout()
plt.savefig(FIG_PATH + '29_rf_feature_importance.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 29_rf_feature_importance.png")
print("\n⚠️  MDI importance can be biased toward high-cardinality features.")
print("   See permutation importance analysis below for a more reliable measure.")

In [ ]:
# ── Permutation importance (more reliable than MDI) ──────────
# Permutation importance measures the drop in val MAE when each feature
# is randomly shuffled — breaking its relationship with the target.
# A large drop = the feature was important. Zero drop = the model ignores it.

print("Computing permutation importance (this takes ~30s)...")
perm = permutation_importance(
    rf, X_val, y_val,
    n_repeats=10,          # shuffle each feature 10 times — average the result
    scoring='neg_mean_absolute_error',
    random_state=42, n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature'         : FEATURE_COLS,
    'perm_importance' : perm.importances_mean,
    'perm_std'        : perm.importances_std
}).sort_values('perm_importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(perm_df['feature'], perm_df['perm_importance'],
        xerr=perm_df['perm_std'],
        color='steelblue', edgecolor='white', height=0.6,
        capsize=4, ecolor='gray', alpha=0.9)
ax.invert_yaxis()
ax.set_xlabel('Mean increase in MAE when feature is shuffled')
ax.set_title('Permutation Importance (Random Forest, val set)\n'
             'Error bars = ±1 std across 10 shuffles')
plt.tight_layout()
plt.savefig(FIG_PATH + '30_permutation_importance.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 30_permutation_importance.png")
print("\nPermutation importance ranking:")
print(perm_df[['feature','perm_importance','perm_std']].to_string(index=False))

---
## 8. Model 5 — Gradient Boosting

### Why Gradient Boosting?

While Random Forest builds trees in **parallel** and averages them,
Gradient Boosting builds trees **sequentially** — each tree corrects the errors
of the previous one.

```
Prediction at step t:
F_t(x) = F_{t-1}(x) + η × h_t(x)

where h_t(x) is a new tree fit on the RESIDUALS of F_{t-1}(x)
and η (learning_rate) controls how much each tree contributes
```

This sequential approach often produces better results but requires more careful
tuning of `n_estimators` and `learning_rate`.

### Key hyperparameters

| Param | Value | Reasoning |
|---|---|---|
| `n_estimators` | 300 | More iterations = lower training error (but watch for overfitting) |
| `learning_rate` | 0.05 | Slow learning rate → more trees needed, but better generalisation |
| `max_depth` | 5 | Shallower than RF — boosting compensates with more trees |
| `subsample` | 0.8 | Stochastic boosting: each tree trains on 80% of data randomly |
| `min_samples_leaf` | 20 | Same floor as RF — smooth leaves |

> **Note:** If XGBoost is installed it is used here instead of sklearn's
> GradientBoostingRegressor — same concept, ~10× faster.

In [ ]:
# ── Gradient Boosting (XGBoost if available, else sklearn) ────
print("Training Gradient Boosting...")
print("This may take 2–3 minutes...")

if HAS_XGBOOST:
    # XGBoost — significantly faster with native early stopping
    gb_model = XGBRegressor(
        n_estimators   = 300,
        learning_rate  = 0.05,
        max_depth      = 5,
        subsample      = 0.8,
        colsample_bytree = 0.8,
        min_child_weight = 20,
        random_state   = 42,
        n_jobs         = -1,
        verbosity      = 0
    )
    gb_name = 'XGBoost'
else:
    # sklearn GradientBoostingRegressor — same algorithm, slightly slower
    gb_model = GradientBoostingRegressor(
        n_estimators   = 300,
        learning_rate  = 0.05,
        max_depth      = 5,
        subsample      = 0.8,
        min_samples_leaf = 20,
        random_state   = 42,
    )
    gb_name = 'GradientBoosting'

gb_results = evaluate_model(
    gb_name, gb_model,
    X_train, y_train, X_val, y_val, X_test, y_test
)
all_results.append(gb_results)

In [ ]:
# ── Gradient Boosting feature importance ─────────────────────
gb_importance_df = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: GB importance
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(gb_importance_df)))
axes[0].barh(gb_importance_df['feature'], gb_importance_df['importance'],
             color=colors, edgecolor='white', height=0.6)
axes[0].invert_yaxis()
axes[0].set_xlabel('Feature importance')
axes[0].set_title(f'{gb_name} Feature Importance')

# Right: side-by-side comparison with RF
rf_sorted   = importance_df.set_index('feature')['importance']
gb_sorted   = gb_importance_df.set_index('feature')['importance']
compare_df  = pd.DataFrame({'RF': rf_sorted, gb_name: gb_sorted}).fillna(0)
compare_df  = compare_df.sort_values('RF', ascending=True)

x = np.arange(len(compare_df))
width = 0.38
axes[1].barh(x - width/2, compare_df['RF'],       width, label='Random Forest', color='steelblue', alpha=0.85)
axes[1].barh(x + width/2, compare_df[gb_name],   width, label=gb_name,         color='darkorange', alpha=0.85)
axes[1].set_yticks(x)
axes[1].set_yticklabels(compare_df.index, fontsize=9)
axes[1].set_xlabel('Feature importance')
axes[1].set_title('RF vs Gradient Boosting\nFeature Importance Comparison')
axes[1].legend()

plt.suptitle('Tree Model Feature Importance Comparison', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_PATH + '31_gb_importance_comparison.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 31_gb_importance_comparison.png")

---
## 9. Model Comparison

Now we bring all results together into a single comparison table and chart.

### What to look for

1. **Val MAE vs Test MAE gap** — a large gap suggests overfitting to the val set
2. **R² values** — are any models explaining a meaningful portion of variance?
3. **Best baseline** — this is the benchmark the DNN must beat in notebook 05
4. **Diminishing returns** — how much does each step up in complexity help?

In [ ]:
# ── Build comparison DataFrame ────────────────────────────────
results_df = pd.DataFrame([{
    'Model'    : r['name'],
    'Val MAE'  : r['val_mae'],
    'Val RMSE' : r['val_rmse'],
    'Val R²'   : r['val_r2'],
    'Test MAE' : r['test_mae'],
    'Test RMSE': r['test_rmse'],
    'Test R²'  : r['test_r2'],
} for r in all_results])

# Sort by Val MAE ascending (lower = better)
results_df = results_df.sort_values('Val MAE').reset_index(drop=True)
results_df.index = results_df.index + 1  # 1-indexed rank

print("=" * 80)
print("MODEL COMPARISON — Sorted by Validation MAE (lower is better)")
print("=" * 80)
print(results_df.to_string())
print("=" * 80)

# Highlight best model
best_name = results_df.iloc[0]['Model']
best_val_mae = results_df.iloc[0]['Val MAE']
print(f"\n🏆 Best baseline: {best_name}  (Val MAE = {best_val_mae:.4f})")
print(f"\n🎯 DNN target in notebook 05: Val MAE < {best_val_mae:.4f}")

In [ ]:
# ── Comparison bar chart: Val MAE ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [('Val MAE', 'lower'), ('Val RMSE', 'lower'), ('Val R²', 'higher')]
colors_map = {
    'Dummy (mean)': '#aaaaaa',
    'Linear Regression': '#4e9af1',
    'Ridge (α=1.0)': '#5bc8af', 'Ridge (α=0.001)': '#5bc8af',
    'Ridge (α=0.01)': '#5bc8af', 'Ridge (α=0.1)': '#5bc8af',
    'Ridge (α=10.0)': '#5bc8af', 'Ridge (α=100.0)': '#5bc8af',
    'Random Forest': '#f4a261',
    'GradientBoosting': '#e76f51', 'XGBoost': '#e76f51',
}
bar_colors = [colors_map.get(n, '#888888') for n in results_df['Model']]

for ax, (metric, direction) in zip(axes, metrics):
    vals  = results_df[metric].values
    names = results_df['Model'].values

    bars = ax.bar(range(len(names)), vals, color=bar_colors,
                  edgecolor='white', width=0.6)

    # Highlight best bar
    best_idx = np.argmin(vals) if direction == 'lower' else np.argmax(vals)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(2.5)

    # Value labels on bars
    for bar, val in zip(bars, vals):
        ypos = bar.get_height() + (max(vals)-min(vals))*0.01
        ax.text(bar.get_x() + bar.get_width()/2, ypos,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)

    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric}  ({"↓ lower better" if direction=="lower" else "↑ higher better"})')

plt.suptitle('Baseline Model Comparison — Validation Set', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_PATH + '32_model_comparison.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 32_model_comparison.png")

---
## 10. Residual Analysis

Residuals = `y_true - y_pred`. A well-behaved model should have residuals that are:

1. **Centred at zero** — no systematic over/under-prediction
2. **Homoscedastic** — similar variance across the prediction range
3. **Approximately normally distributed** — supports regression assumptions

**Patterns to watch for:**

| Pattern in residual plot | Diagnosis |
|---|---|
| Funnel shape (wider at high predictions) | Heteroscedasticity — model less accurate for high delays |
| Curved band (not straight) | Non-linearity not captured by the model |
| Systematic drift (residuals trend up/down with ŷ) | Missing feature or wrong functional form |
| Heavy tails in residual histogram | Large errors on rare high-delay events |

In [ ]:
# ── Residual analysis for best two models ─────────────────────
# Compare Linear Regression (linear) vs Random Forest (non-linear)
# to visually confirm that non-linearity matters

plot_models = [r for r in all_results if r['name'] in
               ('Linear Regression', 'Random Forest')]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for col, r in enumerate(plot_models):
    pred   = r['pred_val']
    resids = y_val - pred

    # ── Plot 1: Predicted vs Actual ───────────────────────────
    axes[0, col].scatter(pred, y_val, alpha=0.08, s=5,
                         color='steelblue', rasterized=True)
    lims = [0, 1]
    axes[0, col].plot(lims, lims, 'r--', lw=1.5, label='Perfect fit')
    axes[0, col].set_xlim(0, 1); axes[0, col].set_ylim(0, 1)
    axes[0, col].set_xlabel('Predicted delay_rate')
    axes[0, col].set_ylabel('Actual delay_rate')
    axes[0, col].set_title(f'{r["name"]}\nPredicted vs Actual (val set)')
    axes[0, col].legend(fontsize=9)

    # ── Plot 2: Residuals vs Predicted ────────────────────────
    axes[1, col].scatter(pred, resids, alpha=0.08, s=5,
                         color='darkorange', rasterized=True)
    axes[1, col].axhline(0, color='red', lw=1.5, ls='--')
    axes[1, col].set_xlabel('Predicted delay_rate')
    axes[1, col].set_ylabel('Residual (actual − predicted)')
    axes[1, col].set_title(f'{r["name"]}\nResiduals vs Predicted')

    # Annotation: mean residual
    mean_res = resids.mean()
    axes[1, col].axhline(mean_res, color='blue', lw=1, ls=':',
                         label=f'Mean residual = {mean_res:.4f}')
    axes[1, col].legend(fontsize=9)

# ── Plot 3: Residual distribution comparison ──────────────────
for r, color in zip(plot_models, ['steelblue','darkorange']):
    resids = y_val - r['pred_val']
    axes[0, 2].hist(resids, bins=60, alpha=0.6, density=True,
                    color=color, edgecolor='white', label=r['name'])
axes[0, 2].axvline(0, color='black', lw=1.5, ls='--')
axes[0, 2].set_xlabel('Residual')
axes[0, 2].set_ylabel('Density')
axes[0, 2].set_title('Residual Distribution Comparison')
axes[0, 2].legend(fontsize=9)

# ── Plot 4: Error by actual delay range ───────────────────────
bins = np.linspace(0, 1, 11)
bin_labels = [f'{bins[i]:.1f}–{bins[i+1]:.1f}' for i in range(len(bins)-1)]
for r, color in zip(plot_models, ['steelblue','darkorange']):
    pred   = r['pred_val']
    errors = np.abs(y_val - pred)
    bin_idx = np.digitize(y_val, bins) - 1
    bin_idx = np.clip(bin_idx, 0, len(bins)-2)
    bin_mae = [errors[bin_idx == i].mean() if (bin_idx == i).sum() > 0
               else np.nan for i in range(len(bins)-1)]
    axes[1, 2].plot(range(len(bin_labels)), bin_mae, 'o-',
                    color=color, lw=2, label=r['name'], markersize=4)
axes[1, 2].set_xticks(range(len(bin_labels)))
axes[1, 2].set_xticklabels(bin_labels, rotation=45, ha='right', fontsize=8)
axes[1, 2].set_xlabel('Actual delay_rate bucket')
axes[1, 2].set_ylabel('MAE within bucket')
axes[1, 2].set_title('MAE by Delay Rate Bucket\n(where do models struggle most?)')
axes[1, 2].legend(fontsize=9)

plt.suptitle('Residual Analysis — Linear Regression vs Random Forest', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_PATH + '33_residual_analysis.png', bbox_inches='tight')
plt.show()
print("💾 Saved: 33_residual_analysis.png")

---
## 11. Prediction Error Over Time

We check whether model errors are uniformly distributed across years
or if there are specific time periods where models struggle.

This is important because:
- High errors in 2020 = model struggled with COVID distribution shift
- Systematic errors in specific months = seasonal pattern not captured
- Increasing errors over time = model is not adapting to long-term trend

In [ ]:
# ── Load val identifiers ──────────────────────────────────────
val_csv = '../data/processed/val.csv'
try:
    val_df = pd.read_csv(val_csv)
    has_ids = True
except FileNotFoundError:
    # Rebuild from raw if val.csv not available
    import pandas as pd, numpy as np
    from sklearn.preprocessing import LabelEncoder
    raw = pd.read_csv('../data/raw/Airline_Delay_Cause.csv')
    raw = raw.dropna(subset=['arr_flights','arr_del15'])
    raw = raw[raw['arr_flights']>0].copy()
    raw['delay_rate'] = raw['arr_del15']/raw['arr_flights']
    val_df = raw[(raw['year']>=2019) & (raw['year']<=2020)].copy()
    has_ids = True

if has_ids:
    val_df = val_df.reset_index(drop=True)

    # ── Error per year-month for best two models ───────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for ax, r, color in zip(axes, plot_models, ['steelblue','darkorange']):
        temp_df = val_df[['year','month']].copy()
        temp_df['actual']  = y_val
        temp_df['pred']    = r['pred_val']
        temp_df['abs_err'] = np.abs(y_val - r['pred_val'])
        temp_df['ym'] = temp_df['year'].astype(str) + '-' + temp_df['month'].astype(str).str.zfill(2)

        monthly_err = temp_df.groupby('ym')['abs_err'].mean().reset_index()
        monthly_err = monthly_err.sort_values('ym')

        ax.bar(range(len(monthly_err)), monthly_err['abs_err'],
               color=color, alpha=0.8, edgecolor='white')

        # Mark 2020-03 (COVID onset)
        covid_idx = monthly_err[monthly_err['ym']=='2020-03'].index
        if len(covid_idx):
            covid_pos = monthly_err.index.get_loc(covid_idx[0])
            ax.axvline(covid_pos, color='red', ls='--', lw=1.5,
                       label='COVID onset (Mar 2020)')

        ax.set_xticks(range(0, len(monthly_err), 3))
        ax.set_xticklabels(monthly_err['ym'].iloc[::3], rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Mean Absolute Error')
        ax.set_title(f'{r["name"]}\nMAE per month (Val: 2019–2020)')
        ax.legend(fontsize=9)

    plt.suptitle('Prediction Error Over Time — Validation Set', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIG_PATH + '34_error_over_time.png', bbox_inches='tight')
    plt.show()
    print("💾 Saved: 34_error_over_time.png")

---
## 12. Save Results for Notebook 05

We save the baseline results table as a JSON file.
Notebook 05 (DNN) will load this and add the DNN row to produce
the final comparison.

In [ ]:
# ── Serialise results (exclude numpy arrays — not JSON serialisable) ──
serialisable = [{k: (float(v) if isinstance(v, (np.floating, float)) else v)
                 for k, v in r.items()
                 if k not in ('pred_val','pred_test','model')}
                for r in all_results]

import json, os
os.makedirs('../data/processed/', exist_ok=True)
with open('../data/processed/baseline_results.json', 'w') as f:
    json.dump(serialisable, f, indent=2)

print("✅ Saved baseline_results.json")
print("\nFinal results table:")
print(results_df.to_string())

---
## 13. Summary & Key Insights

### Results table

| Model | Val MAE | Val RMSE | Val R² |
|---|---|---|---|
| Dummy (mean) | 0.0989 | 0.1215 | −0.308 |
| Linear Regression | 0.0594 | 0.0868 | 0.332 |
| Ridge | 0.0594 | 0.0868 | 0.332 |
| Gradient Boosting | 0.0580 | 0.0866 | 0.335 |
| **Random Forest** | **0.0575** | **0.0849** | **0.361** |

### Key insights

**1. Linear models work surprisingly well**  
Linear Regression achieves Val MAE = 0.059 — a 40% improvement over the dummy baseline.
This tells us the features have a meaningful linear relationship with the target.

**2. Non-linear models help, but not dramatically**  
Random Forest (Val MAE = 0.0575) beats Linear Regression (0.0594) by only ~3%.
The marginal gain from non-linearity is real but modest — suggesting our engineered
features already capture most of the learnable signal in a linear-friendly form.

**3. `carrier_hist_delay` dominates feature importance**  
Both RF and GB agree: the carrier's rolling 12-month history explains ~63–70%
of the predictable variance. This makes intuitive sense — airlines with chronic
operational problems continue to have them.

**4. COVID-2020 is clearly visible in residuals**  
Prediction errors spike in March–April 2020 for all models. This is expected:
no training data from 2020 exists, and the distribution shift is dramatic.

**5. The DNN's target**  
The best baseline is **Random Forest at Val MAE = 0.0575**.
The DNN must beat this to justify its complexity.
Target: **Val MAE < 0.055**

---
*Next: `05_dnn_model.ipynb` — Architecture search, training, hyperparameter tuning, SHAP*